## 波速 Vp 与波阻抗 AI 关系分析

本 notebook 分析 `data/vertical_well_las_target_qyz` 中当前工区井的纵波速度与波阻抗关系。

注意：LAS 中原始曲线是声波时差 `DT/DTCO/DTC`，这里先由 `Vp = 1e6 / DT` 派生纵波速度，再分析 `Vp` 与 `AI` 的关系。本 notebook 只做线性拟合。


In [ ]:
import sys
from pathlib import Path

import lasio
import matplotlib
import numpy as np
import pandas as pd

if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str
    display = print

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.22,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.sans-serif": ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"],
        "axes.unicode_minus": False,
    }
)


In [ ]:
def find_project_root(start: Path) -> Path:
    """Find repo root when the notebook is launched from repo root or notebooks/."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        load_py = candidate / "src" / "cup" / "petrel" / "load.py"
        las_dir = candidate / "data" / "vertical_well_las_target_qyz"
        if load_py.exists() and las_dir.exists():
            return candidate
    raise FileNotFoundError("Cannot find project root containing src/cup/petrel/load.py and target LAS directory.")


project_root = find_project_root(Path.cwd())
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from cup.petrel.load import extract_any_log_from_las

las_dir = project_root / "data" / "vertical_well_las_target_qyz"
las_paths = sorted(las_dir.glob("*.las"))

DT_CANDIDATES = ("DTCO", "DT", "DTC", "AC")
RHO_CANDIDATES = ("RHOZ", "RHOB", "DEN", "HDRA")
SENTINEL_VALUES = (-999.0, -999.25, -99999.0)

DT_UNIT_ASSUMED = "us/m"
RHO_UNIT_ASSUMED = "g/cm3"
EXCLUDED_WELLS: set[str] = {"2-ANP-2A-RJS", "L6-NW3A"}

las_paths = [p for p in las_paths if p.stem not in EXCLUDED_WELLS]

print(f"Project root: {project_root}")
print(f"Excluded wells: {sorted(EXCLUDED_WELLS) if EXCLUDED_WELLS else 'None'}")
print(f"LAS files used: {len(las_paths)}")
for p in las_paths:
    print(f"  - {p.name}")


## 读取与派生规则

- 使用 `lasio.read` 读取 LAS，并使用 `extract_any_log_from_las` 提取原始声波时差和密度曲线。
- 将 `-999`、`-999.25`、`-99999` 和非有限值处理为缺失值。
- 按 `Vp_mps = 1e6 / DT_us_per_m` 由声波时差派生纵波速度。
- 按 `AI = Rho_g_cm3 * Vp_mps` 派生波阻抗。
- 只保留 `Vp`、`AI` 均有效的样点，并只做线性拟合。


In [ ]:
def normalize_mnemonic(name: str) -> str:
    return str(name).strip().upper()


def matches_mnemonic_with_optional_suffix(column_name: str, base_mnemonic: str) -> bool:
    col_norm = normalize_mnemonic(column_name)
    base_norm = normalize_mnemonic(base_mnemonic)
    return col_norm == base_norm or col_norm.startswith(f"{base_norm}_")


def select_curve_mnemonic(las_df: pd.DataFrame, candidates: tuple[str, ...], property_name: str) -> str:
    columns = [str(c) for c in las_df.columns]
    for base in candidates:
        matched = [col for col in columns if matches_mnemonic_with_optional_suffix(col, base)]
        if len(matched) == 1:
            return matched[0]
        if len(matched) > 1:
            raise ValueError(f"{property_name} has multiple candidate curves for {base}: {matched}")
    raise ValueError(f"Cannot find {property_name}. Candidates={candidates}, columns={columns}")


def get_curve_unit(las_file: lasio.LASFile, mnemonic: str) -> str:
    target = normalize_mnemonic(mnemonic)
    for curve in las_file.curves:
        if normalize_mnemonic(curve.mnemonic) == target:
            return str(curve.unit or "")
    return ""


def replace_sentinel_values(values: object) -> np.ndarray:
    out = np.asarray(values, dtype=float).copy()
    for sentinel in SENTINEL_VALUES:
        out[np.isclose(out, sentinel, equal_nan=False)] = np.nan
    out[~np.isfinite(out)] = np.nan
    return out


def extract_raw_log_series(las_file: lasio.LASFile, mnemonic: str) -> pd.Series:
    log = extract_any_log_from_las(las_file, mnemonic)
    return pd.Series(replace_sentinel_values(log.values), index=log.series.index, name=mnemonic)


def linear_fit(x: pd.Series, y: pd.Series) -> tuple[float, float, float]:
    mask = np.isfinite(x.to_numpy(dtype=float)) & np.isfinite(y.to_numpy(dtype=float))
    if mask.sum() < 2:
        return np.nan, np.nan, np.nan
    x_values = x.to_numpy(dtype=float)[mask]
    y_values = y.to_numpy(dtype=float)[mask]
    slope, intercept = np.polyfit(x_values, y_values, deg=1)
    y_pred = slope * x_values + intercept
    ss_res = float(np.sum((y_values - y_pred) ** 2))
    ss_tot = float(np.sum((y_values - np.mean(y_values)) ** 2))
    r2 = np.nan if ss_tot == 0 else 1.0 - ss_res / ss_tot
    return float(slope), float(intercept), float(r2)


def quantile_clip(df: pd.DataFrame, columns: tuple[str, ...], low: float = 0.005, high: float = 0.995) -> pd.DataFrame:
    mask = pd.Series(True, index=df.index)
    for col in columns:
        q_low, q_high = df[col].quantile([low, high])
        mask &= df[col].between(q_low, q_high)
    return df.loc[mask].copy()


In [ ]:
well_frames: list[pd.DataFrame] = []
stats_rows: list[dict[str, object]] = []

for las_path in las_paths:
    las_file = lasio.read(las_path)
    las_df = las_file.df()

    dt_curve = select_curve_mnemonic(las_df, DT_CANDIDATES, "sonic slowness DT")
    rho_curve = select_curve_mnemonic(las_df, RHO_CANDIDATES, "density Rho")

    dt = extract_raw_log_series(las_file, dt_curve)
    rho = extract_raw_log_series(las_file, rho_curve)

    frame = pd.DataFrame(
        {
            "well": las_path.stem,
            "md_m": dt.index.to_numpy(dtype=float),
            "dt_us_per_m": dt.to_numpy(dtype=float),
            "rho_g_cm3": rho.to_numpy(dtype=float),
        }
    )
    frame = frame.replace([np.inf, -np.inf], np.nan)
    frame = frame.dropna(subset=["dt_us_per_m", "rho_g_cm3"]).copy()
    frame = frame.loc[(frame["dt_us_per_m"] > 0) & (frame["rho_g_cm3"] > 0)].copy()
    frame["vp_mps"] = 1_000_000.0 / frame["dt_us_per_m"]
    frame["ai_gcc_mps"] = frame["rho_g_cm3"] * frame["vp_mps"]

    slope, intercept, r2 = linear_fit(frame["vp_mps"], frame["ai_gcc_mps"])
    stats_rows.append(
        {
            "well": las_path.stem,
            "n_rows": len(las_df),
            "dt_curve": dt_curve,
            "dt_unit_raw": get_curve_unit(las_file, dt_curve) or "<empty; assumed us/m>",
            "paired_valid": len(frame),
            "vp_min": frame["vp_mps"].min(),
            "vp_p50": frame["vp_mps"].median(),
            "vp_max": frame["vp_mps"].max(),
            "ai_min": frame["ai_gcc_mps"].min(),
            "ai_p50": frame["ai_gcc_mps"].median(),
            "ai_max": frame["ai_gcc_mps"].max(),
            "pearson_vp_ai": frame["vp_mps"].corr(frame["ai_gcc_mps"], method="pearson"),
            "spearman_vp_ai": frame["vp_mps"].corr(frame["ai_gcc_mps"], method="spearman"),
            "linear_slope_ai_per_mps": slope,
            "linear_intercept": intercept,
            "linear_r2": r2,
        }
    )
    well_frames.append(frame)

all_data = pd.concat(well_frames, ignore_index=True)
summary_df = pd.DataFrame(stats_rows)

rounded_summary = summary_df.copy()
numeric_cols = rounded_summary.select_dtypes(include=[np.number]).columns
rounded_summary[numeric_cols] = rounded_summary[numeric_cols].round(4)
display(rounded_summary)

print(f"Total paired samples: {len(all_data):,}")
print(f"Wells loaded: {summary_df['well'].nunique()}")


In [ ]:
overall_slope, overall_intercept, overall_r2 = linear_fit(all_data["vp_mps"], all_data["ai_gcc_mps"])
overall_pearson = all_data["vp_mps"].corr(all_data["ai_gcc_mps"], method="pearson")
overall_spearman = all_data["vp_mps"].corr(all_data["ai_gcc_mps"], method="spearman")

fit_x = np.linspace(all_data["vp_mps"].quantile(0.005), all_data["vp_mps"].quantile(0.995), 200)
fit_y = overall_slope * fit_x + overall_intercept
plot_data = quantile_clip(all_data, ("vp_mps", "ai_gcc_mps"), low=0.002, high=0.998)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2), constrained_layout=True)

ax = axes[0]
well_names = sorted(plot_data["well"].unique())
colors = plt.cm.tab10(np.linspace(0, 1, len(well_names)))
for color, well in zip(colors, well_names):
    part = plot_data.loc[plot_data["well"] == well]
    ax.scatter(part["vp_mps"], part["ai_gcc_mps"], s=7, alpha=0.32, color=color, label=well, linewidths=0)
ax.plot(fit_x, fit_y, color="black", lw=2.0, label="linear fit")
ax.set_title("全工区 Vp-AI 散点（按井着色）")
ax.set_xlabel("纵波速度 Vp (m/s)")
ax.set_ylabel("波阻抗 AI (g/cm3 * m/s)")
ax.legend(loc="best", fontsize=7, frameon=False, ncols=1)

ax = axes[1]
hb = ax.hexbin(
    plot_data["vp_mps"],
    plot_data["ai_gcc_mps"],
    gridsize=55,
    mincnt=1,
    cmap="viridis",
    bins="log",
)
ax.plot(fit_x, fit_y, color="white", lw=2.0)
ax.set_title("全工区 Vp-AI 样点密度")
ax.set_xlabel("纵波速度 Vp (m/s)")
ax.set_ylabel("波阻抗 AI (g/cm3 * m/s)")
cb = fig.colorbar(hb, ax=ax)
cb.set_label("log10(sample count)")

fig.suptitle(
    f"Vp-AI relationship: Pearson={overall_pearson:.3f}, Spearman={overall_spearman:.3f}, linear R2={overall_r2:.3f}, n={len(all_data):,}",
    fontsize=12,
)
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13.5, 11.0), sharex=True, sharey=True, constrained_layout=True)
axes = axes.ravel()

x_limits = (plot_data["vp_mps"].quantile(0.002), plot_data["vp_mps"].quantile(0.998))
y_limits = (plot_data["ai_gcc_mps"].quantile(0.002), plot_data["ai_gcc_mps"].quantile(0.998))

for ax, well in zip(axes, sorted(all_data["well"].unique())):
    part = all_data.loc[all_data["well"] == well].copy()
    part_plot = quantile_clip(part, ("vp_mps", "ai_gcc_mps"), low=0.002, high=0.998)
    slope, intercept, r2 = linear_fit(part["vp_mps"], part["ai_gcc_mps"])
    x = np.linspace(part_plot["vp_mps"].min(), part_plot["vp_mps"].max(), 100)
    ax.scatter(part_plot["vp_mps"], part_plot["ai_gcc_mps"], s=6, alpha=0.35, linewidths=0)
    ax.plot(x, slope * x + intercept, color="black", lw=1.4)
    pearson = part["vp_mps"].corr(part["ai_gcc_mps"], method="pearson")
    spearman = part["vp_mps"].corr(part["ai_gcc_mps"], method="spearman")
    ax.set_title(f"{well}\nPearson={pearson:.2f}, Spearman={spearman:.2f}, R2={r2:.2f}, n={len(part):,}", fontsize=9)
    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)

for ax in axes[len(all_data["well"].unique()) :]:
    ax.axis("off")

fig.supxlabel("纵波速度 Vp (m/s)")
fig.supylabel("波阻抗 AI (g/cm3 * m/s)")
fig.suptitle("单井 Vp-AI 关系与线性趋势", fontsize=13)
plt.show()


In [ ]:
relationship_table = summary_df[
    [
        "well",
        "paired_valid",
        "pearson_vp_ai",
        "spearman_vp_ai",
        "linear_slope_ai_per_mps",
        "linear_intercept",
        "linear_r2",
        "vp_p50",
        "ai_p50",
    ]
].sort_values("pearson_vp_ai", ascending=False)

display(relationship_table.round(4))

fig, ax = plt.subplots(figsize=(10.5, 4.8), constrained_layout=True)
bars = ax.bar(relationship_table["well"], relationship_table["pearson_vp_ai"], color="#4472C4", alpha=0.82)
ax.axhline(overall_pearson, color="black", lw=1.4, ls="--", label=f"all wells Pearson={overall_pearson:.3f}")
ax.axhline(0, color="0.25", lw=0.8)
ax.set_title("单井 Vp-AI Pearson 相关系数")
ax.set_ylabel("Pearson correlation")
ax.set_xlabel("Well")
ax.tick_params(axis="x", rotation=35)
ax.legend(frameon=False)

for bar, value in zip(bars, relationship_table["pearson_vp_ai"]):
    ax.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.2f}", ha="center", va="bottom" if value >= 0 else "top", fontsize=8)

plt.show()


In [ ]:
weak_or_negative = summary_df.loc[summary_df["pearson_vp_ai"] < 0.70, ["well", "pearson_vp_ai", "spearman_vp_ai"]]
strongest_positive = summary_df.sort_values("pearson_vp_ai", ascending=False).iloc[0]
weakest = summary_df.sort_values("pearson_vp_ai").iloc[0]

vp_q = all_data["vp_mps"].quantile([0.01, 0.5, 0.99])
ai_q = all_data["ai_gcc_mps"].quantile([0.01, 0.5, 0.99])

if weak_or_negative.empty:
    exception_text = "所有井均表现为较强正相关。"
else:
    exception_items = ", ".join(
        f"{row.well} (Pearson={row.pearson_vp_ai:.3f}, Spearman={row.spearman_vp_ai:.3f})"
        for row in weak_or_negative.itertuples(index=False)
    )
    exception_text = f"相关性偏弱的井：{exception_items}。"

conclusion = f"""
## 自动结论

- 当前分析剔除井：**{', '.join(sorted(EXCLUDED_WELLS))}**；实际参与 **{len(las_paths)}** 口井，Vp 与 AI 同时有效的配对样点为 **{len(all_data):,}** 个。
- 全工区 `Vp-AI` 关系整体为正相关：Pearson = **{overall_pearson:.3f}**，Spearman = **{overall_spearman:.3f}**，线性拟合 R2 = **{overall_r2:.3f}**。
- 全工区线性拟合公式：`AI = {overall_slope:.6f} * Vp + {overall_intercept:.4f}`。
- 线性趋势斜率为 **{overall_slope:.4f} AI/(m/s)**。在当前单位下，该斜率可近似理解为由样点共同约束出的有效密度尺度。
- 正相关最强的井为 **{strongest_positive["well"]}**，Pearson = **{strongest_positive["pearson_vp_ai"]:.3f}**；相关性最弱的井为 **{weakest["well"]}**，Pearson = **{weakest["pearson_vp_ai"]:.3f}**。
- {exception_text}
- Vp 的 P01/P50/P99 分别为 **{vp_q.loc[0.01]:.0f} / {vp_q.loc[0.5]:.0f} / {vp_q.loc[0.99]:.0f} m/s**；AI 的 P01/P50/P99 分别为 **{ai_q.loc[0.01]:.0f} / {ai_q.loc[0.5]:.0f} / {ai_q.loc[0.99]:.0f} g/cm3*m/s**。
- 单位假设：LAS 头中 DT 与密度单位字段为空；本分析按 `DT={DT_UNIT_ASSUMED}`、`Rho={RHO_UNIT_ASSUMED}` 处理，并用 `Vp = 1e6 / DT`、`AI = Rho * Vp` 派生速度和波阻抗。
"""

display(Markdown(conclusion))
